# Cinematic Video Generation (Google Colab T4 16GB)

Zero-cost video generation pipeline using Google Colab's free T4 GPU.
- **GPU:** T4 16GB
- **Models:** SDXL (images) + HunyuanVideo 1.5 (video)
- **Fallback:** LTX-Video 2B if HunyuanVideo OOMs

## Instructions
1. Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. Save outputs to Google Drive after EVERY clip

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q huggingface_hub
!pip install -q gfpgan realesrgan basicsr
!pip install -q insightface onnxruntime-gpu

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/video_ai_toolshop'
os.makedirs(f'{DRIVE_DIR}/reference_images', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/raw_video', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/processed_video', exist_ok=True)
print(f'Drive mounted: {DRIVE_DIR}')

In [ ]:
# Cell 3: Configuration
import os

HF_TOKEN = 'hf_your_token_here'
LORA_REPO_ID = 'your_username/my-sdxl-lora'
TRIGGER_WORD = 'ohwx person'

# 10 cinematic scenes
SCENES = [
    {'name': 'neon_tokyo_night', 'prompt': f'{TRIGGER_WORD} walking through neon-lit Tokyo street at night, 35mm anamorphic, volumetric fog, teal and orange grade, photorealistic', 'n': 5},
    {'name': 'golden_hour_field', 'prompt': f'{TRIGGER_WORD} standing in golden wheat field at sunset, 85mm telephoto, warm golden hour lighting, Kodak Portra 400, photorealistic', 'n': 4},
    {'name': 'rainy_city_umbrella', 'prompt': f'{TRIGGER_WORD} holding black umbrella on rainy city street, 50mm, rain droplets, reflections, moody blue tones, photorealistic', 'n': 5},
    {'name': 'studio_portrait', 'prompt': f'{TRIGGER_WORD} dramatic studio portrait, Rembrandt lighting, 135mm, black background, chiaroscuro, photorealistic', 'n': 4},
    {'name': 'futuristic_corridor', 'prompt': f'{TRIGGER_WORD} walking down futuristic corridor with holographic displays, 24mm wide angle, blue purple neon, photorealistic', 'n': 5},
    {'name': 'desert_highway', 'prompt': f'{TRIGGER_WORD} standing beside desert highway at dusk, 35mm anamorphic, dusty atmosphere, warm orange sky, photorealistic', 'n': 4},
    {'name': 'snowy_mountain', 'prompt': f'{TRIGGER_WORD} on snowy mountain peak, bright sunlight, 16mm wide angle, cool blue white palette, photorealistic', 'n': 3},
    {'name': 'jazz_club', 'prompt': f'{TRIGGER_WORD} in dimly lit jazz club, stage spotlight, 50mm, cigarette smoke, warm amber lighting, photorealistic', 'n': 5},
    {'name': 'rooftop_sunrise', 'prompt': f'{TRIGGER_WORD} on city rooftop at sunrise, backlit, 35mm anamorphic, lens flare, warm morning light, photorealistic', 'n': 4},
    {'name': 'subway_station', 'prompt': f'{TRIGGER_WORD} in underground subway station, fluorescent lighting, 24mm, motion blur, green orange grade, photorealistic', 'n': 5},
]

print(f'{len(SCENES)} scenes, {sum(s["n"] for s in SCENES)} total images')

In [ ]:
# Cell 4: Load SDXL + LoRA and generate reference images
import torch
from diffusers import StableDiffusionXLPipeline
from huggingface_hub import hf_hub_download
import json
from datetime import datetime

print('Loading SDXL...')
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16'
).to('cuda')

print('Loading LoRA...')
lora_path = hf_hub_download(repo_id=LORA_REPO_ID, filename='sdxl_lora.safetensors', token=HF_TOKEN)
pipe.load_lora_weights(lora_path)
print(f'LoRA loaded: {lora_path}')

all_metadata = []
for scene in SCENES:
    scene_dir = f'{DRIVE_DIR}/reference_images/{scene["name"]}'
    os.makedirs(scene_dir, exist_ok=True)
    print(f'\n--- {scene["name"]} ({scene["n"]} images) ---')

    for i in range(scene['n']):
        seed = torch.randint(0, 2**32, (1,)).item()
        image = pipe(
            scene['prompt'],
            height=576, width=1024,
            num_inference_steps=30,
            guidance_scale=7.0,
            cross_attention_kwargs={'scale': 0.9},
            generator=torch.Generator('cuda').manual_seed(seed),
        ).images[0]

        filename = f'{scene["name"]}_{i:03d}_seed{seed}.png'
        filepath = f'{scene_dir}/{filename}'
        image.save(filepath)
        all_metadata.append({'filename': filename, 'scene': scene['name'], 'seed': seed, 'prompt': scene['prompt']})
        print(f'  [{i+1}/{scene["n"]}] {filename}')

    # Save metadata
    with open(f'{DRIVE_DIR}/reference_images/generation_metadata.json', 'w') as f:
        json.dump(all_metadata, f, indent=2)

print(f'\n✓ Generated {len(all_metadata)} reference images')
print(f'  Saved to: {DRIVE_DIR}/reference_images/')

# Free VRAM for video generation
del pipe
torch.cuda.empty_cache()

In [ ]:
# Cell 5: Load HunyuanVideo 1.5 and generate video clips
import torch
from diffusers import HunyuanVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import pathlib

print('Loading HunyuanVideo 1.5...')
pipe = HunyuanVideoPipeline.from_pretrained(
    'tencent/HunyuanVideo-1.5',
    torch_dtype=torch.float16,
).to('cuda')
pipe.enable_model_cpu_offload()  # Required for 16GB VRAM
print('HunyuanVideo 1.5 loaded with CPU offloading')

# Find reference images
ref_dir = pathlib.Path(f'{DRIVE_DIR}/reference_images')
ref_images = sorted(ref_dir.rglob('*.png'))
print(f'Found {len(ref_images)} reference images')

# Generate clips using anchor-frame workflow
VIDEO_PROMPT = 'cinematic motion, camera dollying forward, 35mm anamorphic, photorealistic, film grain'
NEGATIVE = 'blurry, distorted, low quality, deformed, watermark, talking, mouth moving'

clips_generated = 0
for i, ref_path in enumerate(ref_images[:10]):  # Start with 10 clips
    print(f'\n--- Clip {i+1}/10: {ref_path.name} ---')

    image = Image.open(ref_path).convert('RGB')
    image = image.resize((1024, 576), Image.LANCZOS)

    try:
        video = pipe(
            image=image,
            prompt=VIDEO_PROMPT,
            num_inference_steps=30,
            num_frames=121,  # 5s @ 24fps
            guidance_scale=7.5,
            negative_prompt=NEGATIVE,
            generator=torch.Generator('cuda').manual_seed(42 + i * 100),
        ).frames[0]

        output_path = f'{DRIVE_DIR}/raw_video/clip_{i+1:03d}.mp4'
        export_to_video(video, output_path, fps=24)
        print(f'  ✓ Saved: {output_path}')
        clips_generated += 1

        # Clear cache between clips
        torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        print(f'  ✗ OOM — try LTX-Video fallback (next cell)')
        torch.cuda.empty_cache()
        break

print(f'\n✓ Generated {clips_generated} clips')
print(f'  Saved to: {DRIVE_DIR}/raw_video/')

In [ ]:
# Cell 6: LTX-Video 2B fallback (if HunyuanVideo OOMs)
# Only run if Cell 5 failed with OOM

import torch
from diffusers import LTXVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import pathlib

print('Loading LTX-Video 2B (8GB VRAM fallback)...')
pipe = LTXVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video-2B',
    torch_dtype=torch.float16,
).to('cuda')
print('LTX-Video loaded')

ref_dir = pathlib.Path(f'{DRIVE_DIR}/reference_images')
ref_images = sorted(ref_dir.rglob('*.png'))

VIDEO_PROMPT = 'cinematic motion, camera dollying forward, 35mm anamorphic, photorealistic'

for i, ref_path in enumerate(ref_images[:10]):
    print(f'\n--- LTX Clip {i+1}: {ref_path.name} ---')
    image = Image.open(ref_path).convert('RGB')

    video = pipe(
        image=image,
        prompt=VIDEO_PROMPT,
        num_inference_steps=30,
        num_frames=121,
        guidance_scale=7.5,
        generator=torch.Generator('cuda').manual_seed(42 + i * 100),
    ).frames[0]

    output_path = f'{DRIVE_DIR}/raw_video/clip_{i+1:03d}.mp4'
    export_to_video(video, output_path, fps=24)
    print(f'  ✓ Saved: {output_path}')
    torch.cuda.empty_cache()

print(f'\n✓ LTX-Video generation complete')

In [ ]:
# Cell 7: Post-processing — GFPGAN → Real-ESRGAN → RIFE
import torch
import cv2
import os
import subprocess
import pathlib

# Install GFPGAN and Real-ESRGAN
!pip install -q gfpgan realesrgan basicsr

from gfpgan import GFPGANer
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# Initialize models
restorer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
    upscale=1,
    arch='clean',
    channel_multiplier=2,
)

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=model,
    tile=512,
    tile_pad=10,
    pre_pad=0,
    half=True,
)

def process_clip(input_path, output_path):
    """Full post-processing chain for one clip."""
    print(f'Processing: {input_path}')

    # Extract frames
    frames_dir = f'/content/frames_{os.path.basename(input_path)}'
    os.makedirs(frames_dir, exist_ok=True)
    subprocess.run(['ffmpeg', '-i', input_path, '-vf', 'fps=24',
                    '-q:v', '2', f'{frames_dir}/frame_%06d.png'], check=True)

    frames = sorted(pathlib.Path(frames_dir).glob('frame_*.png'))
    print(f'  Extracted {len(frames)} frames')

    # GFPGAN + Real-ESRGAN each frame
    processed_dir = f'{frames_dir}_processed'
    os.makedirs(processed_dir, exist_ok=True)

    for frame_path in frames:
        img = cv2.imread(str(frame_path), cv2.IMREAD_UNCHANGED)
        # GFPGAN face restoration
        _, _, restored = restorer.enhance(img, paste_back=True)
        # Real-ESRGAN 4x upscale
        output, _ = upsampler.enhance(restored, outscale=4)
        cv2.imwrite(os.path.join(processed_dir, frame_path.name), output)

    print(f'  Processed {len(frames)} frames')

    # RIFE interpolation via ffmpeg minterpolate (24fps → 48fps)
    output_path_48fps = output_path.replace('.mp4', '_48fps.mp4')
    subprocess.run([
        'ffmpeg', '-framerate', '24',
        '-i', f'{processed_dir}/frame_%06d.png',
        '-vf', 'minterpolate=fps=48:mi_mode=mci:mc_mode=aobmc:me_mode=bidir',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18',
        output_path_48fps
    ], check=True)

    print(f'  ✓ Output: {output_path_48fps}')

    # Cleanup
    import shutil
    shutil.rmtree(frames_dir)
    shutil.rmtree(processed_dir)

# Process all clips
raw_dir = pathlib.Path(f'{DRIVE_DIR}/raw_video')
clips = sorted(raw_dir.glob('*.mp4'))
print(f'Found {len(clips)} clips to process')

for clip in clips:
    output = f'{DRIVE_DIR}/processed_video/{clip.stem}_processed_48fps.mp4'
    process_clip(str(clip), output)
    torch.cuda.empty_cache()

print(f'\n✓ Post-processing complete')